In [1]:
# -----------------------------------------------------------------
# 1. LOAD ORIGINAL RAW DATA
# -----------------------------------------------------------------
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support

INPUT_CSV = "isro_burnin_ess_synthetic_dataset_cleaned.csv"

# Load original raw file without modifying headers
raw_df = pd.read_csv(INPUT_CSV)

work_df = raw_df.copy()
if "component_id" not in work_df.columns and "lot_id" not in work_df.columns:
    work_df = pd.read_csv(INPUT_CSV, skiprows=1)

param_mapping = {
    "iddq_uA": "iddq",
    "leakage_current_nA": "leakage",
    "propagation_delay_ns": "prop_delay",
}

melted_list = []
for prefix, param_name in param_mapping.items():
    sub = pd.DataFrame(
        {
            "lot_id": work_df["lot_id"],
            "part_id": work_df["component_id"],
            "param_name": param_name,
            "value_0h": work_df[f"{prefix}_0h"],
            "value_24h": work_df[f"{prefix}_24h"],
            "value_96h": work_df[f"{prefix}_96h"],
            "value_168h": work_df[f"{prefix}_168h"],
        }
    )
    melted_list.append(sub)

df = pd.concat(melted_list, ignore_index=True)

In [2]:

# -----------------------------------------------------------------
# 2. FEATURE ENGINEERING (SAFE DRIFT RATES)
# -----------------------------------------------------------------
# Calculate hourly drift rates between intervals
df["drift_0_24"] = (df["value_24h"] - df["value_0h"]) / 24.0
df["drift_24_96"] = (df["value_96h"] - df["value_24h"]) / 72.0
df["drift_96_168"] = (df["value_168h"] - df["value_96h"]) / 72.0

# Total relative percentage drift across 168 hours
df["total_drift_pct"] = np.where(
    df["value_0h"] != 0,
    (df["value_168h"] - df["value_0h"]) / df["value_0h"],
    0.0,
)

In [3]:
# -----------------------------------------------------------------
# 3. ROBUST DPAT (Median + MAD, per lot, per parameter)
# -----------------------------------------------------------------
def robust_dpat_limits(series, k=6):
    median = np.median(series)
    mad = np.median(np.abs(series - median))
    sigma_robust = 1.4826 * mad
    upl = median + k * sigma_robust
    lpl = median - k * sigma_robust
    return median, sigma_robust, upl, lpl

def apply_robust_dpat(df, value_col, group_cols=("lot_id", "param_name"), k=6):
    out_rows = []
    for keys, group in df.groupby(list(group_cols)):
        median, sigma, upl, lpl = robust_dpat_limits(group[value_col].values, k=k)
        g = group.copy()
        g[f"{value_col}_median"] = median
        g[f"{value_col}_sigma_robust"] = sigma
        g[f"{value_col}_UPL"] = upl
        g[f"{value_col}_LPL"] = lpl
        g[f"{value_col}_DPAT_outlier"] = (g[value_col] > upl) | (g[value_col] < lpl)
        out_rows.append(g)
    return pd.concat(out_rows, ignore_index=True)

df = apply_robust_dpat(df, "value_0h", k=6)
df = apply_robust_dpat(df, "total_drift_pct", k=6)

In [4]:
# -----------------------------------------------------------------
# 4. ISOLATION FOREST
# -----------------------------------------------------------------
iso_results = []
feature_cols = [
    "value_0h",
    "value_24h",
    "value_96h",
    "value_168h",
    "drift_0_24",
    "drift_24_96",
    "drift_96_168",
    "total_drift_pct",
]

for param, group in df.groupby("param_name"):
    X = group[feature_cols].fillna(0)
    iso = IsolationForest(n_estimators=200, contamination="auto", random_state=42)

    g = group.copy()
    g["iso_forest_flag"] = iso.fit_predict(X) == -1
    g["iso_forest_score"] = iso.decision_function(X)
    iso_results.append(g)

df = pd.concat(iso_results, ignore_index=True)

In [5]:
# -----------------------------------------------------------------
# 5. COMBINE INTO FINAL VERDICT
# -----------------------------------------------------------------
df["dpat_any_flag"] = (
    df["value_0h_DPAT_outlier"] | df["total_drift_pct_DPAT_outlier"]
)

df["final_flag"] = df["dpat_any_flag"] | df["iso_forest_flag"]

df["flag_reason"] = df.apply(
    lambda r: ", ".join(
        filter(
            None,
            [
                "DPAT_0h" if r["value_0h_DPAT_outlier"] else None,
                "DPAT_drift" if r["total_drift_pct_DPAT_outlier"] else None,
                "IsolationForest" if r["iso_forest_flag"] else None,
            ],
        )
    ),
    axis=1,
)

In [6]:
# -----------------------------------------------------------------
# 6. ENRICHED EXPLANATORY OUTPUT REPORT
# -----------------------------------------------------------------
total_evals = len(df)
total_flagged_evals = df["final_flag"].sum()
rejected_part_ids = df[df["final_flag"]]["part_id"].unique()
unique_parts_count = len(rejected_part_ids)

dpat_0h_cnt = df["value_0h_DPAT_outlier"].sum()
dpat_drift_cnt = df["total_drift_pct_DPAT_outlier"].sum()
iso_cnt = df["iso_forest_flag"].sum()
total_components = work_df["component_id"].nunique()

print("=" * 70)
print("             MODULE A: DETAILED SCREENING REPORT")
print("=" * 70)
print(f"1. TOTAL EVALUATIONS CHECKED: {total_evals}")
print(f"2. TOTAL PARAMETER FAILURES FLAGGED: {total_flagged_evals}")
print(f"3. UNIQUE PHYSICAL IC COMPONENTS REJECTED: {unique_parts_count}")
print("=" * 70)

             MODULE A: DETAILED SCREENING REPORT
1. TOTAL EVALUATIONS CHECKED: 30000
2. TOTAL PARAMETER FAILURES FLAGGED: 2081
3. UNIQUE PHYSICAL IC COMPONENTS REJECTED: 948


In [7]:
# -----------------------------------------------------------------
# 7. CONFUSION MATRIX EVALUATION & VISUALIZATION
# -----------------------------------------------------------------
ground_truth_cols = [
    col for col in raw_df.columns
    if col.lower() in ["is_anomaly", "label", "target", "ground_truth"]
]

if ground_truth_cols:
    gt_col = ground_truth_cols[0]
    y_true = work_df[gt_col].astype(int)
    y_pred = work_df["component_id"].isin(rejected_part_ids).astype(int).values
    row_labels = ["Actual Normal (0)", "Actual Outlier (1)"]
    col_labels = ["Predicted Normal (0)", "Predicted Outlier (1)"]
    title = f"MODEL vs GROUND TRUTH ('{gt_col}')"
else:
    y_true = df["dpat_any_flag"].astype(int)
    y_pred = df["iso_forest_flag"].astype(int)
    row_labels = ["DPAT Pass (0)", "DPAT Fail (1)"]
    col_labels = ["IsoForest Pass (0)", "IsoForest Fail (1)"]
    title = "DETECTOR AGREEMENT: ISOLATION FOREST vs STATISTICAL DPAT"

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="binary", zero_division=0
)

cm_df = pd.DataFrame(cm, index=row_labels, columns=col_labels)

print("\n" + "=" * 70)
print(f" {title.center(68)}")
print("=" * 70)
print(cm_df.to_string())
print("-" * 70)
print(f" True Negatives  (TN) : {tn:5d} | Normal items correctly identified")
print(f" False Positives (FP) : {fp:5d} | False alarms (flagged normal items)")
print(f" False Negatives (FN) : {fn:5d} | MISSED OUTLIERS (Not flagged!)")
print(f" True Positives  (TP) : {tp:5d} | Outliers correctly caught")
print("-" * 70)
print(" METRICS SUMMARY:")
print(f"  * Accuracy         : {accuracy * 100:.2f}%")
print(f"  * Precision        : {precision * 100:.2f}%")
print(f"  * Recall           : {recall * 100:.2f}%")
print(f"  * F1-Score         : {f1 * 100:.2f}%")
print(f"  * Missed Outliers  : {fn} outlier(s) missed ({((fn / (fn + tp)) * 100 if (fn + tp) > 0 else 0):.2f}% miss rate)")
print("=" * 70)


       DETECTOR AGREEMENT: ISOLATION FOREST vs STATISTICAL DPAT      
               IsoForest Pass (0)  IsoForest Fail (1)
DPAT Pass (0)               27919                 375
DPAT Fail (1)                 122                1584
----------------------------------------------------------------------
 True Negatives  (TN) : 27919 | Normal items correctly identified
 False Positives (FP) :   375 | False alarms (flagged normal items)
 False Negatives (FN) :   122 | MISSED OUTLIERS (Not flagged!)
 True Positives  (TP) :  1584 | Outliers correctly caught
----------------------------------------------------------------------
 METRICS SUMMARY:
  * Accuracy         : 98.34%
  * Precision        : 80.86%
  * Recall           : 92.85%
  * F1-Score         : 86.44%
  * Missed Outliers  : 122 outlier(s) missed (7.15% miss rate)


In [8]:
# -----------------------------------------------------------------
# 8. EXPORT CSV FILES
# -----------------------------------------------------------------
# Save evaluation audit log
df.to_csv("module_a_output.csv", index=False)

# Identify original ID column header
id_col = (
    "component_id"
    if "component_id" in raw_df.columns
    else raw_df.columns[raw_df.isin(rejected_part_ids).any()].tolist()[0]
)

# Export clean CSV preserving exact original table structure & headers
clean_raw_df = raw_df[~raw_df[id_col].isin(rejected_part_ids)].copy()
clean_raw_df.to_csv("clean_burnin_dataset.csv", index=False)

print("\nFiles successfully created:")
print(" -> module_a_output.csv (Evaluation audit log)")
print(" -> clean_burnin_dataset.csv (Exact original CSV structure without outliers)")


Files successfully created:
 -> module_a_output.csv (Evaluation audit log)
 -> clean_burnin_dataset.csv (Exact original CSV structure without outliers)
